In [1]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

# 🕵️ Rewriting Biographies

Instead of replacing entities with tokens, rewrite mode generates a
privacy-safe paraphrase of the entire text. The pipeline:

1. Detects entities (same as replace mode)
2. Classifies the domain and assigns sensitivity dispositions
3. Generates a rewritten version that obscures sensitive entities
4. Evaluates quality (utility) and privacy (leakage) with an automated repair loop
5. Runs a final LLM judge for informational scores

#### 📚 What you'll learn

- Configure rewrite mode with `PrivacyGoal` to specify what to protect and what to preserve
- Set evaluation criteria and risk tolerance for automated quality checks
- Preview rewritten text and inspect utility / leakage scores
- Triage flagged records with `needs_human_review`

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
- Import `Rewrite` and its config classes: `PrivacyGoal`, `EvaluationCriteria`, `RiskTolerance`.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.

In [2]:
import getpass
import os

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
    os.environ["NVIDIA_API_KEY"] = key

In [3]:
from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, Rewrite
from anonymizer.config.rewrite import EvaluationCriteria, PrivacyGoal, RiskTolerance

In [4]:
anonymizer = Anonymizer()

[18:23:31] [INFO] 🔧 Anonymizer initialized with 3 model configs


[18:23:31] [INFO]   |-- 🔎 detector:  gliner-pii-detector


[18:23:31] [INFO]   |-- ✅ validator: gpt-oss-120b


[18:23:31] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 📦 Input data

- Same biographies dataset used in earlier notebooks -- familiar data makes it
  easy to compare rewrite output against replace output.

In [5]:
input_data = AnonymizerInput(
    source="../data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles",
)

## 🎛️ Configure

- `PrivacyGoal` spells out what to **protect** and what to **preserve** --
  this gives the rewriter clear, domain-specific guidance.
- `EvaluationCriteria` controls the automated quality gate: `risk_tolerance`
  sets the leakage threshold and `max_repair_iterations` caps how many times
  the rewriter retries when evaluation fails.

In [6]:
config = AnonymizerConfig(
    rewrite=Rewrite(
        privacy_goal=PrivacyGoal(
            protect="All direct identifiers and quasi-identifier combinations (names, locations, employers, dates)",
            preserve="Career trajectory, educational background, and professional accomplishments",
        ),
        evaluation=EvaluationCriteria(
            risk_tolerance=RiskTolerance.strict,
            max_repair_iterations=3,
        ),
    ),
)

## 👁️ Preview

- `preview()` runs on a small sample so you can iterate on privacy goals
  and evaluation criteria before committing to a full run.

In [7]:
preview = anonymizer.preview(
    config=config,
    data=input_data,
    num_records=3,
)

preview.display_record(0)

[18:23:31] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:23:31] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:23:31] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:23:31] [INFO] 🔍 Running entity detection on 3 records


[18:24:40] [INFO]   |-- 📋 Detection complete — 74 entities found across 3 records (0 failed) [69.4s]


...

[18:28:04] [INFO] Evaluate-repair loop iteration 1: 1/3 rows need repair


[18:28:39] [INFO] Evaluate-repair loop: all rows pass at iteration 2


[18:28:53] [INFO]   |-- 📋 Rewrite complete (0 failed) [252.8s]


[18:28:53] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Entity,Label,Sensitivity,Protection
40,age,medium,generalize
Bobby,first_name,high,replace
Mexican,race_ethnicity,high,generalize
veterinarian,occupation,low,left_as_is
Denver,city,medium,generalize
Colorado,state,low,left_as_is
Jefferson High,school_name,medium,replace
DVM,education_level,low,left_as_is
University of Colorado Boulder,university_name,medium,replace
English,language,low,left_as_is


In [8]:
preview.display_record(1)

Entity,Label,Sensitivity,Protection
37,age,medium,generalize
Bell,last_name,high,replace
Edison,city,medium,generalize
Elena,first_name,low,replace
English,language,low,left_as_is
Goddard Space Flight Center,company_name,medium,replace
Idilio,first_name,high,replace
Italian,race_ethnicity,low,left_as_is
Lina,first_name,low,replace
Marco,first_name,low,replace


## 🚀 Full run

- `result.dataframe` has user-facing columns: rewritten text, scores, and the review flag.
- `result.trace_dataframe` has every intermediate column for debugging.

In [9]:
result = anonymizer.run(config=config, data=input_data)

result.dataframe.head()

[18:28:53] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:28:53] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:28:53] [INFO] 🔍 Running entity detection on 25 records


[18:35:23] [INFO]   |-- 📋 Detection complete — 629 entities found across 25 records (0 failed) [390.0s]


[18:35:23] [INFO]   |-- labels: first_name=154, company_name=71, city=52, occupation=44, education_level=37, state=34, race_ethnicity=30, last_name=27, age=26, religious_belief=26, political_view=25, street_address=23, language=21, employment_status=12, county=12, date_of_birth=9, date=5, organization_name=4, region=3, university_name=3, college_name=2, education_institution=2, school_name=1, country=1, professional_body=1, government_agency=1, gender=1, institution_name=1, postcode=1


...

[18:57:24] [INFO] Evaluate-repair loop iteration 1: 5/25 rows need repair


[18:58:22] [INFO] Evaluate-repair loop iteration 2: 3/25 rows need repair


/Users/lramaswamy/Documents/github/Anonymizer/src/anonymizer/engine/rewrite/rewrite_workflow.py:86: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(list(parts), ignore_index=True)
[18:59:32] [INFO]   |-- 📋 Rewrite complete (0 failed) [1449.1s]


[18:59:32] [INFO] 🎉 Pipeline complete — 25 records processed, 0 total failures


,biography,biography_rewritten,utility_score,leakage_mass,any_high_leaked,needs_human_review
0,"Bobby Watford, a 40‑year‑old Mexican veterinar...","Ethan Khan, in his early 40s, Mexican veterina...",0.92,0.6,False,False
1,Idilio Bell is a 37‑year‑old astronomer living...,Luca Harper is a late‑30s astronomer residing ...,0.725,0.0,False,False
2,"Jodi Allison, 36, lives at 204 Bluegrass in Cl...","Leah Harper, in their late 30s, lives at 317 W...",0.875,0.6,False,False
3,James Mills is a 69‑year‑old paramedic who liv...,Robert Harper is a paramedic in his late 60s w...,0.745455,0.0,False,False
4,Nancy Burton is a 21‑year‑old cashier who live...,Sofia Hawkins is in her early twenties and wor...,0.955556,0.6,False,False


In [10]:
result.dataframe[["biography_rewritten", "utility_score", "leakage_mass", "needs_human_review"]].head()

,biography_rewritten,utility_score,leakage_mass,needs_human_review
0,"Ethan Khan, in his early 40s, Mexican veterina...",0.92,0.6,False
1,Luca Harper is a late‑30s astronomer residing ...,0.725,0.0,False
2,"Leah Harper, in their late 30s, lives at 317 W...",0.875,0.6,False
3,Robert Harper is a paramedic in his late 60s w...,0.745455,0.0,False
4,Sofia Hawkins is in her early twenties and wor...,0.955556,0.6,False


In [11]:
result.trace_dataframe.columns.tolist()

['biography',
 '_anonymizer_record_id',
 '_raw_detected_entities',
 '_seed_entities',
 '_initial_tagged_text',
 '_seed_entities_json',
 '_tag_notation',
 '_merged_tagged_text',
 '_validation_candidates',
 '_validated_entities',
 '_augmented_entities',
 '_merged_entities',
 '_detected_entities',
 'biography_with_spans',
 '_latent_entities',
 'final_entities',
 '_entities_by_value',
 '_entity_examples',
 '_entities_for_replace',
 '_entities_for_replace_json',
 '_replacement_map',
 '_domain',
 '_domain_supplement',
 '_sensitivity_disposition',
 '_sensitivity_disposition_block',
 '_privacy_qa',
 '_rewrite_disposition_block',
 '_meaning_units',
 '_replacement_map_for_prompt',
 '_meaning_units_serialized',
 '_full_rewrite',
 '_quality_qa',
 'biography_rewritten',
 '_repair_iterations',
 '_quality_qa_reanswer',
 '_privacy_qa_reanswer',
 '_quality_qa_compare',
 'utility_score',
 'leakage_mass',
 'any_high_leaked',
 '_needs_repair',
 '_leaked_privacy_items',
 '_rewritten_text__next',
 '_judge_e

## 🚩 Filter by review flag

- Records where automated metrics exceed thresholds are flagged for manual review.
- Use this to prioritize human attention on the records that need it most.

In [12]:
df = result.dataframe
flagged = df[df["needs_human_review"] == True]  # noqa: E712
print(f"{len(flagged)} of {len(df)} records flagged for human review")
flagged.head()

2 of 25 records flagged for human review


,biography,biography_rewritten,utility_score,leakage_mass,any_high_leaked,needs_human_review
13,"Shane Wile, 61, lives at 7 Mounds Rd in Cooper...","Trevor Keller, in his early 60s, lives at 9 Va...",0.825,1.0,True,True
18,Reza Amir is a 65‑year‑old carpenter who lives...,Dariush Haddad is a man in his 60s who works a...,0.9375,2.4,False,True


## ⏭️ Next steps

- **[⚖️ Rewriting Legal Documents](05_rewriting_legal_documents.ipynb)** --
  rewrite legal text with custom entity labels and domain-specific privacy goals.
- **[🎯 Choosing a Replacement Strategy](03_choosing_a_replacement_strategy.ipynb)** --
  compare Redact, Annotate, Hash, and Substitute if you prefer token-level replacement.
- **[🔍 Inspecting Detected Entities](02_inspecting_detected_entities.ipynb)** --
  debug what the detection pipeline found before rewriting.